# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

The dataset describes protein abundance, modifications, and peptide sequences in human (Homo sapiens) samples, including various attributes such as accession, coverage percentage, peptide counts, molecular weight, and calculated properties (e.g., pI, normalized abundances).

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an object, not a dict

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their `@id` fields.

In [ ]:
# List available record sets and their fields with corresponding @id
print("Available record sets (by @id):")
for recset in dataset.record_sets:
    print(f"- RecordSet @id: {recset.id} (name: {recset.name})")
    print("  Fields:")
    for field in recset.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print()
# For illustration, print a sample record for the first record set
if dataset.record_sets:
    example_record_set = dataset.record_sets[0]
    print(f"First 2 records from record set {example_record_set.id}:")
    for i, row in enumerate(dataset.records(record_set=example_record_set.id)):
        print(row)
        if i == 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the `@id` of the record set and field(s) identified above. This notebook loads all record sets.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

# Extract data from each record set as pandas DataFrame
dataframes = {}
for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"Loaded RecordSet @id: {recset_id} --> Shape: {df.shape}")

# Display sample columns and preview the first record set
if record_set_ids:
    preview_id = record_set_ids[0]
    print(f"Columns in RecordSet {preview_id}:")
    print(dataframes[preview_id].columns.tolist())
    display(dataframes[preview_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate:
- Filtering records based on a numeric field (e.g., coverage percentage or peptide count)
- Normalizing a numeric field
- Grouping by a categorical field (e.g., describing the protein)

Fields are always referenced by their `@id` as per the Croissant schema.

In [ ]:
# For demonstration, select the first record set and numeric fields found in overview above.
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id] if record_set_id else None
print(f"Working on RecordSet @id: {record_set_id}")

# Identify a numeric field (by @id) for analysis
# Let's heuristically pick the first field with type integer/float
numeric_field_id = None
for field in dataset.record_set(record_set_id).fields:
    dt = (field.data_type or "").lower()
    if "float" in dt or "int" in dt or "number" in dt:
        numeric_field_id = field.id
        break
if numeric_field_id is None and df is not None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
print(f"Selected numeric field (by @id): {numeric_field_id}")

# Filtering step
if df is not None and numeric_field_id in df.columns:
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt to group by the first string/categorical field
        group_field_id = None
        for field in dataset.record_set(record_set_id).fields:
            if field.id != numeric_field_id and field.data_type == 'Text':
                group_field_id = field.id
                break
        if not group_field_id:
            # Fallback: find first non-numeric col
            for col in df.columns:
                if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field_id:
                    group_field_id = col
                    break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped means of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    except Exception as e:
        print(f"Error during EDA: {e}")

## 5. Visualization

Visualize the distribution of the selected numeric field and, if available, the grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouped means were created above, plot them
    if 'grouped_df' in locals() and group_field_id and grouped_df is not None:
        plt.figure(figsize=(8,4))
        topn = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)
        sns.barplot(data=topn, x=numeric_field_id, y=group_field_id, orient='h')
        plt.title(f"Top 20 groups by mean {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook demonstrated how to:
- Load the mass spectrometry dataset Croissant schema via `mlcroissant`,
- Identify record sets and fields by their `@id`s,
- Extract data for analysis using Pandas,
- Perform basic data filtering, normalization, grouping, and visualization.

For more detailed analysis, consult the field definitions in the [FAIR^2 Croissant package](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and refer to the `mlcroissant` documentation for advanced usage.